<a href="https://colab.research.google.com/github/poorvikulkarni3/Plant-Disease-detection-cnn/blob/main/Plant_Disease_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Seeding for reproducibility

In [1]:
import random
random.seed(0)

import numpy as np
np.random.seed(0)

import tensorflow as tf
tf.random.set_seed(0)

OS library is used to interact with os, create files, folders, directories etc

zipfiles to unzip files

PIL for image proecessing like resizing

keras - high level api by tensorflow for building cnn

Tensorflow - open source ml

In [2]:
import os
from zipfile import ZipFile
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

**Data Curation**


In [3]:
!pip install -q -U kaggle     #kaggle already exists but -u is used to update it here

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 11.9 MB/s eta 0:00:00


This is API authentication and this token(plant disease) is used to communicate with Kaggle

In [4]:
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("Plant")

In [5]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [02:00<00:00, 18.2MB/s]



In [6]:
!ls

plantvillage-dataset.zip  sample_data


To unzip the dataset

In [7]:
with ZipFile("plantvillage-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [8]:
print(os.listdir("plantvillage dataset"))    #to list the directories in the unzipped dataset


print(len(os.listdir("plantvillage dataset/segmented")))     #lists the number
print(os.listdir("plantvillage dataset/segmented")[:5])      #lists the names inside this folder

print(len(os.listdir("plantvillage dataset/color")))
print(os.listdir("plantvillage dataset/color")[:5])

print(len(os.listdir("plantvillage dataset/grayscale")))
print(os.listdir("plantvillage dataset/grayscale")[:5])

['segmented', 'grayscale', 'color']
38
['Potato___Late_blight', 'Raspberry___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Grape___healthy', 'Apple___Apple_scab']
38
['Potato___Late_blight', 'Raspberry___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Grape___healthy', 'Apple___Apple_scab']
38
['Potato___Late_blight', 'Raspberry___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Grape___healthy', 'Apple___Apple_scab']


Colour subfolder has 38 sub folders which futher contain plant name and disease names

In [9]:
print(len(os.listdir("plantvillage dataset/color/Grape___healthy")))
print(os.listdir("plantvillage dataset/color/Grape___healthy")[:5])

423
['bfd1909d-6235-41a4-a24a-eeea9ffdd0a5___Mt.N.V_HL 9042.JPG', '0ec29818-c70e-4586-9481-3201a67b7910___Mt.N.V_HL 6175.JPG', '6b11a8e2-a729-4f80-9836-e90402c0b47f___Mt.N.V_HL 6062.JPG', 'bfeeb99b-946a-4edb-b59e-445d682d891d___Mt.N.V_HL 6061.JPG', '18db578d-ab6c-4469-8e07-01d8c211a91f___Mt.N.V_HL 8983.JPG']


**Data Preprocessing**

We are going with colour subfolder for our model. Hence we are creating a base directory with that dataset path

In [10]:
base_dir = 'plantvillage dataset/color'

In [11]:
# Image Parameters
img_size = 224    #resizes the images to this dimension
batch_size = 32    #number of images used to training at once

**Train Test Split**

Keras splits into 2 subsets only - hence we are dividing this subset into training subset and validation subset

Imagedatagenerator - a keras utility for building and loading images in batches and augmenting them for preprocessing

In [12]:
# Image Data Generators
data_gen = ImageDataGenerator(
    rescale=1./255,   #normalisation- used to get pixel values between 0 and 1 i.e a gaussian distribution
    validation_split=0.2  # Use 20% of data for validation and 80% for testing
)

flowfromdirectory looks into the dataset and assigns class from the folder names and groups images into batches - useful for multiclass classification

In [13]:
# Train Generator
train_generator = data_gen.flow_from_directory(
    base_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    subset='training',
    class_mode='categorical'      #for one hot encoding
)

Found 43456 images belonging to 38 classes.


In [14]:
# Validation Generator
validation_generator = data_gen.flow_from_directory(
    base_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    subset='validation',
    class_mode='categorical'
)

Found 10849 images belonging to 38 classes.


**Convolutional Neural Network**

In [15]:
# Model Definition
model = models.Sequential()   #sequential is a class in the models module of keras. it is used to build cnn in a sequential manner(layer after layer)

model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(img_size, img_size, 3)))    #performs convolution. used 32 filters/kernels which slides over the image producing a feature map
#shape is a function which returns the size #relu is a function which makes all -ve values 0 and positive values linear
model.add(layers.MaxPooling2D(2, 2))    #.add is a method used to specify what the next layer is in the sequence
#max pooling layer uses (2,2) block on the feature map and in this block we take max value. stride for max pooling is taken as 2 so it moves 2 steps ahead which reduces the size of feature map
#and reduces computation

model.add(layers.Conv2D(64, (3, 3), activation='relu'))     #convolution but with 64 layers
model.add(layers.MaxPooling2D(2, 2))

model.add(layers.Flatten())     #we are flattening the feature map to a 1D vector for the dense layer to process it
model.add(layers.Dense(256, activation='relu'))    #dense layer which is fully connected neuron layer with 256 neurons? why 256? conventional sizing used usually. using 512 makes it harder during
#computation
model.add(layers.Dense(train_generator.num_classes, activation='softmax'))  #output layer where train gen which gives 38 classes and we are not hardcoding this 38. 38 number of outputs are possible
#using softmax activation because its multi class

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
# model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    47,776,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,805,158 (182.36 MB)

 Trainable params: 47,805,158 (182.36 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Compile the Model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])